# Llenado de plan de produccion
Este proceso consolida la informacion del archivo Tracker para llenar el plan de produccion sin alterar las formulas de excel
- V1. 2025-02-05
    - Version Inicial

## Seleccion de archivos

In [215]:
# Seleccion de archivos de trabajo
import pandas as pd
import os
import ipywidgets as widgets
from ipywidgets import  Layout,HTML
from IPython.display import display, Markdown, clear_output
import pickle
from datetime import datetime, timedelta
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Alignment, Font
from openpyxl.formatting.rule import CellIsRule
from tqdm.notebook import tqdm
from tkinter import Tk, filedialog as fd
import win32com.client
import warnings
from copy import copy 
import gc

# Funciones generales
warnings.filterwarnings("ignore")
def open_file_selection(initialdir='',filter_name='*.*'):
    """
    Opens a file selection dialog and returns the selected file paths.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: A tuple of selected file paths or an empty tuple if no file is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top
    
    filetypes = (
        (filter_name,f"*{filter_name.split(' ')[0]}*.*"),
        ('All files', '*.*'),
        ('Excel', '*.xlsx'),
        ('CSV', '*.doc')
    )
    files = fd.askopenfilenames(
        filetypes=filetypes,
        initialdir=initialdir
    )
    
    # Destroy the root window after use
    root.destroy()
    
    return files
def select_directory(initialdir):
    """
    Opens a directory selection dialog and returns the selected directory path.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: The selected directory path as a string, or an empty string if no directory is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top

    # Open the directory selection dialog
    directory = fd.askdirectory(initialdir=initialdir)

    # Destroy the root window after use
    root.destroy()

    return directory

# Function to show a pop-up message
def show_popup_message(message):
    display(Markdown(f"### **{message}**"))
    

# Decorator to handle the permission error
def handle_permission_error_with_popup(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PermissionError as e:
            if e.errno == 13:  # Permission denied error
                show_popup_message(f"Error: {e}\nFavor de cerrar el archivo.")
    return wrapper

# @handle_permission_error_with_popup
# def save_df(df, filepath,sheet_name,index):
#     df.to_excel(filepath,sheet_name=sheet_name,index=index)

def save_df(df, filepath, sheet_name, index=False):
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index)

@handle_permission_error_with_popup
def save_df_multiple(df_dict=dict(), filepath='',index=False):
    with pd.ExcelWriter(filepath) as writer:
        for key in df_dict.keys():
            df_dict[key].to_excel(writer, sheet_name=key, index=index)

@handle_permission_error_with_popup
def save_wb(wb, filepath):
    wb.save(filepath)

@handle_permission_error_with_popup
def append_sheet(df,path,sheet_name,index):
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index) 


def read_excel(path=None,sheet_name=0,header=0):
    with pd.ExcelFile(path) as xls:
        df=pd.read_excel(path,sheet_name=sheet_name,header=header)
    return df

def is_file_open(filepath):
    # Check if the file exists
    if not os.path.exists(filepath):
        return False  # File does not exist, so treat it as "open" for your logic

    try:
        # Try to open the file in write mode
        with open(filepath, 'a'):
            pass
        return False  # File is not open (no exception raised)
    except PermissionError:
        return True 
    
def verify_selections(file_selectors):
    not_selected=[]
    # not_selected=[file_selector.children[0].description[0:-1] for file_selector in file_selectors if (file_selector.children[1].value.strip()=='Not selected') and (file_selector.children[0].description[0:-1] in mandatory_selectors)]
    for file_selector in file_selectors:
        selector=file_selector.children[0].description[0:-1]
        selected=file_selector.children[1].value.strip()
        if not os.path.exists(selected):
            file_selector.children[1].value='Not selected'
            selected='Not selected'
        if selector not in mandatory_selectors:
            continue
        if selected=='Not selected':
            not_selected.append(selector)
            continue

    if len(not_selected)>0:
        show_popup_message(f'Favor de seleccionar los siguientes archivos:{not_selected}')
        raise SystemExit()

def close_xl_if_open(path):
    if is_file_open(path):
        try:
            excel = win32com.client.Dispatch("Excel.Application")
            workbook = excel.Workbooks(path)
            workbook.Save()
            workbook.Close()
        except:
            show_popup_message(f"Cerrar el archivo: {path}")
            raise SystemExit()     
def check_mandatory_cols(cols,selector_name):
    missing_columns = [col for col in mandatory_cols[selector_name] if col not in cols]
    if len(missing_columns)>0:
        show_popup_message(f"No se encontraron las siguientes columnas en el archivo {selector_name}: {missing_columns}")
        raise SystemExit()
    return

def save_state_pickle(state, filename='folder_state.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(state, f)

def load_state_pickle(filename='folder_state.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}

def on_output_button_click(b):
    if state['folder_output']:
        initialdir=state['folder_output']
    else:
        initialdir='/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_output_label.value = f"{selected_dir}"
        state['folder_output']=selected_dir
        save_state_pickle(state)
    else:
        folder_output_label.value = "Not selected"
    set_paths(folder_output_label.value)

def format_dates(df, date_cols=[]):
    for col in date_cols:
        df[col]=pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    return df


def set_paths(path):
    """
    Set global paths
    """
    global output_paths
    output_paths={}
    output_paths['path_prod_report']=os.path.join(path, "Plan de Produccion.xlsx")
    output_paths['path_reports']=os.path.join(path,'Valores no encontrados.xlsx')


def get_path(file_selectors,selector_name):
    file_selector=[file_selector for file_selector in file_selectors if file_selector.children[0].description[0:-1]==selector_name]
    if len(file_selector)==0:
        show_popup_message(f"No se encontro el selector")
        raise SystemExit()
    file_selector=file_selector[0]
    if not file_selector.children[1].value:
        return None
    selected_path=file_selector.children[1].value.strip()
    return selected_path

def standardize_columns(df, column_mapping):
    """Estandariza las columnas de un DataFrame según un diccionario de mapeo."""
    reverse_mapping = {orig: standard for standard, cols in column_mapping.items() for orig in cols}
    return df.rename(columns=reverse_mapping)

def inverse_standardize_columns(df, column_mapping, columns_to_replace):
    """Reverts standard column names to original names based on column_mapping."""
    inverse_mapping = {standard: orig for standard, cols in column_mapping.items() for orig in cols if orig in columns_to_replace}
    return df.rename(columns=inverse_mapping)
   

# Excel management

def find_cell_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                return cell.coordinate  # Return cell address (e.g., 'B2')
    return None 
def get_locations(df,ws,column):
    """
    Obtains all the cell locations in a worksheet (ws) for each text in a column of a dataframe
    """
    coord={}
    for text in df[column].drop_duplicates():
        coord[text]=find_cell_by_text(ws,text)
        coord
    return coord

def get_cell_properties(cell):
    properties = {}
    # Get the fill color (background)
    fill = cell.fill
    if fill and fill.fgColor and fill.fgColor.type == "rgb":
        rgb = fill.fgColor.rgb  # ARGB format
        properties["background_color"] = f"{rgb[2:]}"  # Skip alpha
    else:
        properties["background_color"] = None
    # Get the font color (text color)
    font_color = cell.font.color
    if font_color and font_color.type == "rgb":
        rgb = font_color.rgb  # ARGB format
        properties["text_color"] = f"{rgb[2:]}"  # Skip alpha
    else:
        properties["text_color"] = 'FFFFFF'
    properties["font_name"]=cell.font.name
    properties["font_size"]=cell.font.size
    # Check if the content is wrapped
    properties["is_wrapped"] = cell.alignment.wrap_text if cell.alignment else False
    # Check if the text is bold
    properties["is_bold"] = cell.font.bold if cell.font else False
    return properties

def get_column_info(ws, col_name):
    """
    Returns a dictionary with the column cell and the data range for the column.
    """
    col_cell=find_cell_by_text(ws, col_name)
    if not col_cell:
        show_popup_message(f"Column '{col_name}' not found in the sheet.")
        raise SystemExit()
    col_cell=ws[col_cell]
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    data_range=ws[col_cell.offset(1,0).coordinate:ws.cell(ws[last_cell].row,col_cell.offset(1,0).column).coordinate]
    data_range_list=[cell[0] for cell in data_range]
    return {'data_range':data_range_list,
            'col_cell':col_cell}

def format_cell(cell,properties):
    "Apply properties based on the dict of prooperties"
    if properties['background_color']:
        cell.fill = PatternFill(start_color=properties['background_color'], end_color=properties['background_color'], fill_type="solid")
    else:
        cell.fill = PatternFill(fill_type=None)
    cell.font = Font(name=properties['font_name'],size=properties['font_size'],color=properties['text_color'], bold=properties['is_bold'])
    cell.alignment = Alignment(wrap_text=properties['is_wrapped'])


# Configuraciones

# Agregar las columnas criticas por archivo
mandatory_cols={
    'Tracker':[
        'PO cliente',
        'Modelo',
        'WO\n QTY',
        'START DATE',
        'REPROGRAMACION',
        'FINISH DATE',
        'SHIP DATE'
                            ]
                }
mandatory_selectors=[
    'Tracker','Plan de Produccion']

filters = ['Tracker','Plan de Produccion']

search_parms={
    "column":'Date',
    "row":'PN',
    "offset":
            {"Category":
            {
            "scheduled":(0,0),
            "real":(1,0)
            },
            "Shift":
            {"1s":(0,0),
            "2s":(0,2)},
            }
}




column_mapping={
    'PO':['Customer PO#','PO cliente','PurchaseOrder'],
    'Modelo':['Producto','ProductServiceID','Part No.'],
    'Quantity':['WO\n QTY','Cantidad','PO Quantity'],
    'ShipmentDate':['CUU ship Date'],
    'PODate':['EDI Received'],
    'WO':['Work Order'],  
    'SOUS': ['REF02_CustomerOrderNumber']  
}

wo_plan_dt='REPROGRAMACION'
wo_result_dt='FINISH DATE'
wo_family_col='Famlia'
wo_status_col='Status Kit'


search_parms={
    "column":'Dates',
    "row":wo_family_col,
    "offset":
            {"date_type":
            {
            "REPROGRAMACION":(0,0),
            "FINISH DATE":(1,0)
            }
            }
}

# UI
state = load_state_pickle()
folder_output_button = widgets.Button(description="Folder de salidas:")
if state['folder_output']:
    initialdir=state['folder_output']
    set_paths(initialdir)
else:
    initialdir='Not selected'
folder_output_label = widgets.Label(value=initialdir)

folder_output_button.on_click(on_output_button_click)

# Create an array of button and label widgets
file_selectors = []
for filter_name in filters:
    # Create button and label
    button = widgets.Button(description=f"{filter_name}:")
    if state.get(filter_name):
        label = widgets.Label(value=f" {state.get(filter_name)}")
    else:
        label = widgets.Label(value=f" Not selected")
    
    # Define the button click event
    def on_button_click(filter_name=filter_name, label=label):
        if (filter_name in state.keys()) and (state[filter_name]):
            initialdir=os.path.dirname(state[filter_name][0])
        else:
            initialdir='/'
        selected_dir = open_file_selection(initialdir=initialdir,filter_name=filter_name)  # Adjust initialdir as needed
        if selected_dir:
            label.value = f" {selected_dir[0]}"
            state[filter_name]=selected_dir[0]
            
        else:
            label.value = f" Not selected"
            state[filter_name]=f" Not selected"
        save_state_pickle(state)

    # Attach the event to the button
    button.on_click(lambda b, f=filter_name, l=label: on_button_click(f, l))
    
    # Add the button and label as a horizontal box
    file_selectors.append(widgets.HBox([button, label]))



ui = widgets.VBox([widgets.HBox([folder_output_button, folder_output_label])]+file_selectors)
display(ui)

for file_selector in file_selectors:
    filter_name=file_selector.children[0].description[:-1]


In [222]:
# Verificar selecciones
verify_selections(file_selectors)

In [217]:
# Consolidar reporte de work orders
path_tracker=get_path(file_selectors,'Tracker')
df_wo_wb=pd.read_excel(path_tracker,sheet_name=None)
df_wo=pd.DataFrame()
for sheet in df_wo_wb.keys():
    if 'Plan de produccion' in sheet:
        df_wo=pd.concat([df_wo,df_wo_wb[sheet]])
        df_wo.dropna(subset=['PO cliente','Modelo'],inplace=True)
        


check_mandatory_cols(df_wo.columns,'Tracker')
df_wo=standardize_columns(df_wo,column_mapping)
df_wo['Modelo']=df_wo['Modelo'].str.upper()
df_data = df_wo.melt(id_vars=[wo_family_col, 'Quantity'], value_vars=[wo_plan_dt, wo_result_dt],
                    var_name='date_type', value_name='Dates')
df_data=format_dates(df_data,['Dates'])
df_data=df_data.groupby(['Dates',wo_family_col,'date_type']).sum()
df_data.reset_index(inplace=True)
df_data['Dates']=pd.to_datetime(df_data['Dates'])

# Se obtiene el formato y prioridad de los diferentes status
path_prod_repo=get_path(file_selectors,'Plan de Produccion')
close_xl_if_open(path_prod_repo)
wb=load_workbook(path_prod_repo)
ws=wb['CC']
col_info=get_column_info(ws,'Status')
status_cell_format={}
for cell in col_info['data_range']:
    status_cell_format[cell.value]=get_cell_properties(cell)
df_status=read_excel(path_prod_repo,sheet_name='CC')

# El status define por fecha planeada
df_wo_status=df_wo[[wo_family_col,wo_plan_dt,wo_status_col]].drop_duplicates()
df_wo_status.rename({wo_status_col:'Status'},axis=1,inplace=True)
df_wo_status=df_wo_status.merge(df_status,how='left',on='Status')
df_wo_status.fillna(0,inplace=True)
df_wo_status.sort_values('Prioridad',inplace=True)
df_wo_status.drop_duplicates(subset=[wo_family_col,wo_plan_dt],keep='first',inplace=True)
df_wo_status=df_wo_status[df_wo_status['Prioridad']>0]

In [219]:
# Llenar el formato

ws=wb['PLAN - RESULTS']
df_data_coord=df_data[[search_parms['column'],search_parms['row']]].drop_duplicates()
df_data_coord['row']=''
df_data_coord['column']=''
coord_y=get_locations(df_data_coord,ws,search_parms['row'])
coord_x=get_locations(df_data_coord,ws,search_parms['column'])
for index,row in df_data.iterrows():
    if not ((coord_x[row[search_parms['column']]] and coord_y[row[search_parms['row']]])):
        continue  
    family=row[search_parms['row']]  
    date=row[search_parms['column']]
    cell=ws.cell(ws[coord_y[family]].row,
                                            ws[coord_x[date]].column)
    for key in search_parms['offset'].keys():
        if not key in df_data.columns:
            continue
        offset=search_parms['offset'][key][row[key]]
        cell=cell.offset(offset[0],offset[1])
    cell.value=row['Quantity']
    df=df_wo_status[(df_wo_status[wo_family_col]==family) & (df_wo_status[wo_plan_dt]==date)]
    if not df.empty:
        status=df.iloc[0]['Status']
        format_cell(cell,status_cell_format[status])
wb.save(path_prod_repo)

In [221]:
# Reporte de datos faltantes
close_xl_if_open(output_paths['path_reports'])
df_dict={}
df_dts = pd.DataFrame(list(coord_x.items()), columns=['Date', 'Cell'])
df_dts=df_dts[df_dts['Cell'].isnull()][['Date']]
if not df_dts.empty:
    df_dict['Fechas Faltantes']=df_dts
df_fams = pd.DataFrame(list(coord_y.items()), columns=['Familias', 'Cell'])
df_fams=df_fams[df_fams['Cell'].isnull()][['Familias']]
if not df_fams.empty:
    df_interleaved = pd.DataFrame(
    "",
    index=range(2 * len(df_fams)),
    columns=df_fams.columns
    )
    df_interleaved.iloc[0::2, :] = df_fams.values
    df_interleaved.reset_index(drop=True, inplace=True)
    df_dict['Familias Faltantes']=df_interleaved
if df_dict:
    save_df_multiple(df_dict,filepath=output_paths['path_reports'])
    os.startfile(output_paths['path_reports'])
else:
    os.remove(output_paths['path_reports'])
os.startfile(path_prod_repo)